In [ ]:
# 1. Install packages (restart kernel after the first install: Kernel -> Restart Kernel)
%pip install -q xgboost joblib

In [ ]:
# 2a. Write features.py (shared feature definitions; the web app uses the same file)
FEATURES_SRC = '"""\nWallet features, computed the same way as the Elliptic++ "actors" dataset.\n\nEach Elliptic++ row describes one wallet address during one time step (~2 weeks,\nabout 2016 Bitcoin blocks). We rebuild the same features from live blockchain.com\ndata so the model sees live wallets exactly the way it saw training wallets.\n"""\nimport numpy as np\n\nSATOSHI = 1e8\nSTATS = [\'total\', \'min\', \'max\', \'mean\', \'median\']\n\n# per-transaction series whose total/min/max/mean/median become features\nGROUPS = [\'btc_transacted\', \'btc_sent\', \'btc_received\', \'fees\', \'fees_as_share\',\n          \'blocks_btwn_txs\', \'blocks_btwn_input_txs\', \'blocks_btwn_output_txs\',\n          \'transacted_w_address\']\n\nSINGLES = [\'num_txs_as_sender\', \'num_txs_as_receiver\', \'total_txs\',\n           \'lifetime_in_blocks\', \'num_addr_transacted_multiple\']\n\nALL_FEATURES = SINGLES + [f\'{g}_{s}\' for g in GROUPS for s in STATS]\n\n# plain-English names for reports\nLABELS = {\n    \'num_txs_as_sender\': \'transactions sending funds\',\n    \'num_txs_as_receiver\': \'transactions receiving funds\',\n    \'total_txs\': \'total transactions\',\n    \'lifetime_in_blocks\': \'active lifetime (blocks)\',\n    \'num_addr_transacted_multiple\': \'counterparties dealt with more than once\',\n    \'btc_transacted\': \'BTC moved per transaction\',\n    \'btc_sent\': \'BTC sent per transaction\',\n    \'btc_received\': \'BTC received per transaction\',\n    \'fees\': \'fees paid per transaction (BTC)\',\n    \'fees_as_share\': \'wallet share of fees (BTC)\',\n    \'blocks_btwn_txs\': \'blocks between transactions\',\n    \'blocks_btwn_input_txs\': \'blocks between sends\',\n    \'blocks_btwn_output_txs\': \'blocks between receipts\',\n    \'transacted_w_address\': \'interactions per counterparty\',\n}\n\n\ndef label(feature):\n    """\'blocks_btwn_txs_median\' -> \'median blocks between transactions\'."""\n    for g in GROUPS:\n        if feature.startswith(g + \'_\'):\n            stat = feature[len(g) + 1:]\n            return f"{stat} {LABELS[g]}" if stat != \'total\' else f"total {LABELS[g].replace(\' per transaction\', \'\')}"\n    return LABELS.get(feature, feature)\n\n\ndef _stats(values, prefix):\n    v = np.asarray(values, dtype=float)\n    if v.size == 0:\n        return {f\'{prefix}_{s}\': 0.0 for s in STATS}\n    return {f\'{prefix}_total\': float(v.sum()), f\'{prefix}_min\': float(v.min()),\n            f\'{prefix}_max\': float(v.max()), f\'{prefix}_mean\': float(v.mean()),\n            f\'{prefix}_median\': float(np.median(v))}\n\n\ndef _gaps(heights):\n    h = sorted(heights)\n    return np.diff(h) if len(h) > 1 else []\n\n\ndef tx_view(tx, address):\n    """What one transaction looks like from the wallet\'s point of view."""\n    inputs, outs = tx.get(\'inputs\', []), tx.get(\'out\', [])\n    in_addrs = {i.get(\'prev_out\', {}).get(\'addr\') for i in inputs}\n    out_addrs = {o.get(\'addr\') for o in outs}\n    sent = sum(i.get(\'prev_out\', {}).get(\'value\', 0) for i in inputs\n               if i.get(\'prev_out\', {}).get(\'addr\') == address) / SATOSHI\n    received = sum(o.get(\'value\', 0) for o in outs if o.get(\'addr\') == address) / SATOSHI\n    is_input, is_output = address in in_addrs, address in out_addrs\n    # counterparties follow the money: when sending -> the recipients; when receiving -> the senders\n    # (co-senders / co-recipients in the same transaction are NOT counterparties)\n    others = (out_addrs if is_input else set()) | (in_addrs if is_output else set())\n    others = {a for a in others if a and a != address}\n    fee = tx.get(\'fee\', 0) / SATOSHI\n    tx_in_total = sum(i.get(\'prev_out\', {}).get(\'value\', 0) for i in inputs) / SATOSHI\n    tx_total = tx_in_total or (sum(o.get(\'value\', 0) for o in outs) / SATOSHI + fee)\n    return {\'height\': tx[\'block_height\'], \'sent\': sent, \'received\': received,\n            \'is_input\': is_input, \'is_output\': is_output, \'fee\': fee, \'tx_total\': tx_total,\n            \'others\': others}\n\n\ndef compute_features(txs, address):\n    """Elliptic++-style features for one wallet over a list of confirmed transactions."""\n    views = [tx_view(t, address) for t in txs if t.get(\'block_height\') is not None]\n    if not views:\n        return {f: 0.0 for f in ALL_FEATURES}\n\n    heights = [v[\'height\'] for v in views]\n    moved = [v[\'sent\'] + v[\'received\'] for v in views]\n    counterparty_counts = {}\n    for v in views:\n        for a in v[\'others\']:\n            counterparty_counts[a] = counterparty_counts.get(a, 0) + 1\n\n    f = {\n        \'num_txs_as_sender\': float(sum(v[\'is_input\'] for v in views)),\n        \'num_txs_as_receiver\': float(sum(v[\'is_output\'] for v in views)),\n        \'total_txs\': float(len(views)),\n        \'lifetime_in_blocks\': float(max(heights) - min(heights)),\n        \'num_addr_transacted_multiple\': float(sum(c > 1 for c in counterparty_counts.values())),\n    }\n    f.update(_stats(moved, \'btc_transacted\'))\n    # Elliptic++ averages sent/received over ALL the wallet\'s transactions (0 where it didn\'t send/receive)\n    f.update(_stats([v[\'sent\'] for v in views], \'btc_sent\'))\n    f.update(_stats([v[\'received\'] for v in views], \'btc_received\'))\n    f.update(_stats([v[\'fee\'] for v in views], \'fees\'))\n    # the wallet\'s share of each fee: fee x (wallet\'s amount / whole transaction value), in BTC\n    f.update(_stats([v[\'fee\'] * m / v[\'tx_total\'] for v, m in zip(views, moved) if v[\'tx_total\'] > 0],\n                    \'fees_as_share\'))\n    f.update(_stats(_gaps(heights), \'blocks_btwn_txs\'))\n    f.update(_stats(_gaps([v[\'height\'] for v in views if v[\'is_input\']]), \'blocks_btwn_input_txs\'))\n    f.update(_stats(_gaps([v[\'height\'] for v in views if v[\'is_output\']]), \'blocks_btwn_output_txs\'))\n    # how many times the wallet dealt with each counterparty (one value per counterparty)\n    f.update(_stats(list(counterparty_counts.values()), \'transacted_w_address\'))\n    return f\n\n\ndef windowed_features(txs, address, window_blocks=2016, cumulative=False):\n    """\n    Split a wallet\'s history into ~2-week windows (like Elliptic time steps) and compute\n    features per window. cumulative=True uses all history up to each window\'s end,\n    matching the training data if Elliptic++ rows turn out to be cumulative.\n    """\n    confirmed = sorted([t for t in txs if t.get(\'block_height\') is not None],\n                       key=lambda t: t[\'block_height\'])\n    if not confirmed:\n        return []\n    start = confirmed[0][\'block_height\']\n    buckets = {}\n    for t in confirmed:\n        buckets.setdefault((t[\'block_height\'] - start) // window_blocks, []).append(t)\n    out = []\n    for k in sorted(buckets):\n        w_start = start + k * window_blocks\n        w_end = w_start + window_blocks - 1\n        window_txs = [t for t in confirmed if t[\'block_height\'] <= w_end] if cumulative else buckets[k]\n        out.append({\'block_start\': w_start, \'block_end\': w_end,\n                    \'n_txs\': len(buckets[k]), \'features\': compute_features(window_txs, address)})\n    return out\n'
open('features.py', 'w').write(FEATURES_SRC)
print('features.py written')

In [ ]:
# 2. Imports + settings
import os, json, time, sys
import numpy as np, pandas as pd, boto3, joblib, requests
import sklearn, xgboost
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve, auc, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from features import ALL_FEATURES, SINGLES, GROUPS, STATS, compute_features

BUCKET    = 'sagemaker-notebook-469627110704'
OUT       = 'elliptic/wallet_model'      # where the model bundle goes in S3
DATA_DIR  = 'data_pp'
SEED      = 42
s3 = boto3.client('s3')
np.random.seed(SEED)

In [ ]:
# 3. Download Elliptic++ wallet files from S3 (found anywhere in the bucket) and load them
os.makedirs(DATA_DIR, exist_ok=True)
keys = [o['Key'] for p in s3.get_paginator('list_objects_v2').paginate(Bucket=BUCKET) for o in p.get('Contents', [])]
for fn in ['wallets_features.csv', 'wallets_classes.csv']:
    local = f'{DATA_DIR}/{fn}'
    if not os.path.exists(local):
        match = [k for k in keys if k.endswith(fn)]
        if not match:
            raise FileNotFoundError(f'{fn} not in bucket. Files there: {keys}')
        s3.download_file(BUCKET, match[0], local); print('downloaded', match[0])

wf = pd.read_csv(f'{DATA_DIR}/wallets_features.csv')
wc = pd.read_csv(f'{DATA_DIR}/wallets_classes.csv')
print(wf.shape, wc.shape)
print(list(wf.columns))
print(wc['class'].value_counts())

In [ ]:
# 4. Map dataset column names to our feature names (features.py) and attach labels
def canonical(col):
    n = col.strip().lower().replace(' ', '_')
    if n in SINGLES or n in ('address', 'time_step', 'first_block_appeared_in', 'last_block_appeared_in'):
        return n
    for s in STATS:
        if n.endswith('_' + s):
            base = n[:-len(s) - 1]
            if base.startswith('fees_as_share') or base.startswith('fees_share'): g = 'fees_as_share'
            elif base.startswith('fees'):                g = 'fees'
            elif base.startswith('btc_transacted'):      g = 'btc_transacted'
            elif base.startswith('btc_sent'):            g = 'btc_sent'
            elif base.startswith('btc_received'):        g = 'btc_received'
            elif 'btwn_input' in base:                   g = 'blocks_btwn_input_txs'
            elif 'btwn_output' in base:                  g = 'blocks_btwn_output_txs'
            elif 'btwn_tx' in base:                      g = 'blocks_btwn_txs'
            elif base.startswith('transacted_w_address'): g = 'transacted_w_address'
            else: return n
            return f'{g}_{s}'
    return n

wf = wf.rename(columns={c: canonical(c) for c in wf.columns})
FEATURES = [f for f in ALL_FEATURES if f in wf.columns]
missing = [f for f in ALL_FEATURES if f not in wf.columns]
print(f'using {len(FEATURES)} features; missing: {missing}')

wc.columns = ['address', 'class']
wc['y'] = wc['class'].astype(str).map({'1': 1, '2': 0}).fillna(-1).astype(int)   # 1 illicit, 2 licit, 3 unknown
df = wf.merge(wc[['address', 'y']], on='address', how='left')
df['y'] = df['y'].fillna(-1).astype(int)
df = df[df['y'] != -1].reset_index(drop=True)
print('labeled rows:', len(df), '| addresses:', df['address'].nunique())
print(df.groupby('y')['address'].nunique().rename({0: 'licit', 1: 'illicit'}))

In [ ]:
# 5. Diagnostics: is each row one time step, or cumulative history? (decides how the app builds live features)
per_step = df.groupby(['address', 'time_step']).size()
print('rows per (address, time step): max', per_step.max(), '| mean', round(per_step.mean(), 2))
multi = df.sort_values('time_step').groupby('address').filter(lambda g: g['time_step'].nunique() > 1)
nondecreasing = (multi.groupby('address')['total_txs'].apply(lambda s: s.is_monotonic_increasing).mean()
                 if len(multi) else float('nan'))
print('share of multi-step addresses whose total_txs never decreases:', round(nondecreasing, 3))
print('lifetime_in_blocks 50/95/99th pct:', df['lifetime_in_blocks'].quantile([.5, .95, .99]).round().tolist())
CUMULATIVE = bool(nondecreasing > 0.97 and df['lifetime_in_blocks'].quantile(.95) > 4000)
print('CUMULATIVE =', CUMULATIVE)

In [ ]:
# 6. Temporal split by the time step where each address FIRST appears (no address in two splits)
first_step = df.groupby('address')['time_step'].min()
split = pd.cut(first_step, [0, 29, 34, 99], labels=['train', 'val', 'test'])
df['split'] = df['address'].map(split)
for s in ['train', 'val', 'test']:
    part = df[df.split == s]
    print(f"{s:5s} rows {len(part):7,} | addresses {part.address.nunique():6,} | illicit rows {part.y.mean():.1%}")

tr, va, te = (df.split == 'train'), (df.split == 'val'), (df.split == 'test')
X, y = df[FEATURES].astype('float32'), df['y'].values
pos_weight = (y[tr] == 0).sum() / (y[tr] == 1).sum()
print('pos_weight', round(pos_weight, 2))

In [ ]:
# 7. Train XGBoost and Random Forest; score at the ADDRESS level (max risk over its rows = how the app scores)
def address_level(mask, probs):
    g = pd.DataFrame({'address': df.loc[mask, 'address'].values, 'p': probs, 'y': y[mask]}).groupby('address')
    return g['y'].max().values, g['p'].max().values

def best_threshold(yt, p):
    pr, rc, th = precision_recall_curve(yt, p)
    f1 = 2 * pr[:-1] * rc[:-1] / np.clip(pr[:-1] + rc[:-1], 1e-9, None)
    return float(th[np.argmax(f1)])

def metrics(yt, p, thr):
    pred = (p >= thr).astype(int); pr, rc, _ = precision_recall_curve(yt, p)
    return {'Illicit F1': f1_score(yt, pred), 'Illicit Recall': recall_score(yt, pred),
            'Illicit Precision': precision_score(yt, pred, zero_division=0), 'PR-AUC': auc(rc, pr), 'threshold': thr}

use_gpu = False
try:
    import torch; use_gpu = torch.cuda.is_available()
except Exception:
    pass

models = {
    'XGBoost': XGBClassifier(n_estimators=1500, learning_rate=0.05, max_depth=6, subsample=0.8,
                             colsample_bytree=0.8, scale_pos_weight=pos_weight, eval_metric='aucpr',
                             early_stopping_rounds=50, tree_method='hist',
                             device='cuda' if use_gpu else 'cpu', random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_leaf=5, max_samples=0.5,
                                            class_weight='balanced_subsample', n_jobs=-1, random_state=SEED),
}
results, val_ap = {}, {}
for name, m in models.items():
    t0 = time.time()
    if name == 'XGBoost':
        m.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
    else:
        m.fit(X[tr], y[tr])
    yv, pv = address_level(va, m.predict_proba(X[va])[:, 1])
    yt, pt = address_level(te, m.predict_proba(X[te])[:, 1])
    thr = best_threshold(yv, pv)
    results[name] = metrics(yt, pt, thr)
    val_ap[name] = metrics(yv, pv, thr)['PR-AUC']
    print(f'{name}: trained in {time.time() - t0:.0f}s | val PR-AUC {val_ap[name]:.3f}')

table = pd.DataFrame(results).T.round(4)
print(table)
BEST = max(val_ap, key=val_ap.get)          # chosen on VALIDATION, not test
print('selected model:', BEST)

In [ ]:
# 8. Reference statistics for explanations + out-of-range warnings (training rows only)
model, thr = models[BEST], results[BEST]['threshold']
train_rows = df[tr]
ref_stats = {f: {'licit_median':   float(train_rows.loc[train_rows.y == 0, f].median()),
                 'illicit_median': float(train_rows.loc[train_rows.y == 1, f].median()),
                 'p01': float(train_rows[f].quantile(.01)), 'p99': float(train_rows[f].quantile(.99))}
             for f in FEATURES}
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(importances.head(10).round(4))

In [ ]:
# 9. Save the model bundle + pinned app requirements + demo addresses, upload to S3
if BEST == 'XGBoost':
    model.set_params(device='cpu')          # the EC2 web server has no GPU
bundle = {'model': model, 'model_name': BEST, 'features': FEATURES, 'threshold': thr,
          'cumulative': CUMULATIVE, 'window_blocks': 2016, 'ref_stats': ref_stats,
          'importance_order': list(importances.index), 'metrics': results,
          'versions': {'sklearn': sklearn.__version__, 'xgboost': xgboost.__version__,
                       'numpy': np.__version__, 'pandas': pd.__version__, 'python': sys.version.split()[0]}}
joblib.dump(bundle, 'wallet_model.joblib', compress=3)

with open('requirements_app.txt', 'w') as f:
    f.write(f"scikit-learn=={sklearn.__version__}\nxgboost=={xgboost.__version__}\n"
            f"numpy=={np.__version__}\npandas=={pd.__version__}\njoblib\nstreamlit\nrequests\nboto3\n")

p_te = model.predict_proba(X[te])[:, 1]
test_addr = pd.DataFrame({'address': df.loc[te, 'address'].values, 'p': p_te, 'y': y[te]}) \
              .groupby('address').agg(p=('p', 'max'), y=('y', 'max'), rows=('p', 'size')).reset_index()
demo = pd.concat([test_addr[(test_addr.y == 1) & (test_addr.p >= thr)].nlargest(10, 'rows').assign(kind='illicit (caught)'),
                  test_addr[(test_addr.y == 0) & (test_addr.p < thr)].nsmallest(10, 'p').assign(kind='licit')])
print(demo['kind'].value_counts())
demo.to_csv('demo_addresses.csv', index=False)
table.to_csv('wallet_results.csv')

for fn in ['wallet_model.joblib', 'requirements_app.txt', 'demo_addresses.csv', 'wallet_results.csv']:
    s3.upload_file(fn, BUCKET, f'{OUT}/{fn}'); print(f'uploaded s3://{BUCKET}/{OUT}/{fn}')
print('model size MB:', round(os.path.getsize('wallet_model.joblib') / 1e6, 1))

In [ ]:
# 10. Write the web app files and upload them to S3 (EC2 downloads them from here)
APP_FILES = {'features.py': '"""\nWallet features, computed the same way as the Elliptic++ "actors" dataset.\n\nEach Elliptic++ row describes one wallet address during one time step (~2 weeks,\nabout 2016 Bitcoin blocks). We rebuild the same features from live blockchain.com\ndata so the model sees live wallets exactly the way it saw training wallets.\n"""\nimport numpy as np\n\nSATOSHI = 1e8\nSTATS = [\'total\', \'min\', \'max\', \'mean\', \'median\']\n\n# per-transaction series whose total/min/max/mean/median become features\nGROUPS = [\'btc_transacted\', \'btc_sent\', \'btc_received\', \'fees\', \'fees_as_share\',\n          \'blocks_btwn_txs\', \'blocks_btwn_input_txs\', \'blocks_btwn_output_txs\',\n          \'transacted_w_address\']\n\nSINGLES = [\'num_txs_as_sender\', \'num_txs_as_receiver\', \'total_txs\',\n           \'lifetime_in_blocks\', \'num_addr_transacted_multiple\']\n\nALL_FEATURES = SINGLES + [f\'{g}_{s}\' for g in GROUPS for s in STATS]\n\n# plain-English names for reports\nLABELS = {\n    \'num_txs_as_sender\': \'transactions sending funds\',\n    \'num_txs_as_receiver\': \'transactions receiving funds\',\n    \'total_txs\': \'total transactions\',\n    \'lifetime_in_blocks\': \'active lifetime (blocks)\',\n    \'num_addr_transacted_multiple\': \'counterparties dealt with more than once\',\n    \'btc_transacted\': \'BTC moved per transaction\',\n    \'btc_sent\': \'BTC sent per transaction\',\n    \'btc_received\': \'BTC received per transaction\',\n    \'fees\': \'fees paid per transaction (BTC)\',\n    \'fees_as_share\': \'wallet share of fees (BTC)\',\n    \'blocks_btwn_txs\': \'blocks between transactions\',\n    \'blocks_btwn_input_txs\': \'blocks between sends\',\n    \'blocks_btwn_output_txs\': \'blocks between receipts\',\n    \'transacted_w_address\': \'interactions per counterparty\',\n}\n\n\ndef label(feature):\n    """\'blocks_btwn_txs_median\' -> \'median blocks between transactions\'."""\n    for g in GROUPS:\n        if feature.startswith(g + \'_\'):\n            stat = feature[len(g) + 1:]\n            return f"{stat} {LABELS[g]}" if stat != \'total\' else f"total {LABELS[g].replace(\' per transaction\', \'\')}"\n    return LABELS.get(feature, feature)\n\n\ndef _stats(values, prefix):\n    v = np.asarray(values, dtype=float)\n    if v.size == 0:\n        return {f\'{prefix}_{s}\': 0.0 for s in STATS}\n    return {f\'{prefix}_total\': float(v.sum()), f\'{prefix}_min\': float(v.min()),\n            f\'{prefix}_max\': float(v.max()), f\'{prefix}_mean\': float(v.mean()),\n            f\'{prefix}_median\': float(np.median(v))}\n\n\ndef _gaps(heights):\n    h = sorted(heights)\n    return np.diff(h) if len(h) > 1 else []\n\n\ndef tx_view(tx, address):\n    """What one transaction looks like from the wallet\'s point of view."""\n    inputs, outs = tx.get(\'inputs\', []), tx.get(\'out\', [])\n    in_addrs = {i.get(\'prev_out\', {}).get(\'addr\') for i in inputs}\n    out_addrs = {o.get(\'addr\') for o in outs}\n    sent = sum(i.get(\'prev_out\', {}).get(\'value\', 0) for i in inputs\n               if i.get(\'prev_out\', {}).get(\'addr\') == address) / SATOSHI\n    received = sum(o.get(\'value\', 0) for o in outs if o.get(\'addr\') == address) / SATOSHI\n    is_input, is_output = address in in_addrs, address in out_addrs\n    # counterparties follow the money: when sending -> the recipients; when receiving -> the senders\n    # (co-senders / co-recipients in the same transaction are NOT counterparties)\n    others = (out_addrs if is_input else set()) | (in_addrs if is_output else set())\n    others = {a for a in others if a and a != address}\n    fee = tx.get(\'fee\', 0) / SATOSHI\n    tx_in_total = sum(i.get(\'prev_out\', {}).get(\'value\', 0) for i in inputs) / SATOSHI\n    tx_total = tx_in_total or (sum(o.get(\'value\', 0) for o in outs) / SATOSHI + fee)\n    return {\'height\': tx[\'block_height\'], \'sent\': sent, \'received\': received,\n            \'is_input\': is_input, \'is_output\': is_output, \'fee\': fee, \'tx_total\': tx_total,\n            \'others\': others}\n\n\ndef compute_features(txs, address):\n    """Elliptic++-style features for one wallet over a list of confirmed transactions."""\n    views = [tx_view(t, address) for t in txs if t.get(\'block_height\') is not None]\n    if not views:\n        return {f: 0.0 for f in ALL_FEATURES}\n\n    heights = [v[\'height\'] for v in views]\n    moved = [v[\'sent\'] + v[\'received\'] for v in views]\n    counterparty_counts = {}\n    for v in views:\n        for a in v[\'others\']:\n            counterparty_counts[a] = counterparty_counts.get(a, 0) + 1\n\n    f = {\n        \'num_txs_as_sender\': float(sum(v[\'is_input\'] for v in views)),\n        \'num_txs_as_receiver\': float(sum(v[\'is_output\'] for v in views)),\n        \'total_txs\': float(len(views)),\n        \'lifetime_in_blocks\': float(max(heights) - min(heights)),\n        \'num_addr_transacted_multiple\': float(sum(c > 1 for c in counterparty_counts.values())),\n    }\n    f.update(_stats(moved, \'btc_transacted\'))\n    # Elliptic++ averages sent/received over ALL the wallet\'s transactions (0 where it didn\'t send/receive)\n    f.update(_stats([v[\'sent\'] for v in views], \'btc_sent\'))\n    f.update(_stats([v[\'received\'] for v in views], \'btc_received\'))\n    f.update(_stats([v[\'fee\'] for v in views], \'fees\'))\n    # the wallet\'s share of each fee: fee x (wallet\'s amount / whole transaction value), in BTC\n    f.update(_stats([v[\'fee\'] * m / v[\'tx_total\'] for v, m in zip(views, moved) if v[\'tx_total\'] > 0],\n                    \'fees_as_share\'))\n    f.update(_stats(_gaps(heights), \'blocks_btwn_txs\'))\n    f.update(_stats(_gaps([v[\'height\'] for v in views if v[\'is_input\']]), \'blocks_btwn_input_txs\'))\n    f.update(_stats(_gaps([v[\'height\'] for v in views if v[\'is_output\']]), \'blocks_btwn_output_txs\'))\n    # how many times the wallet dealt with each counterparty (one value per counterparty)\n    f.update(_stats(list(counterparty_counts.values()), \'transacted_w_address\'))\n    return f\n\n\ndef windowed_features(txs, address, window_blocks=2016, cumulative=False):\n    """\n    Split a wallet\'s history into ~2-week windows (like Elliptic time steps) and compute\n    features per window. cumulative=True uses all history up to each window\'s end,\n    matching the training data if Elliptic++ rows turn out to be cumulative.\n    """\n    confirmed = sorted([t for t in txs if t.get(\'block_height\') is not None],\n                       key=lambda t: t[\'block_height\'])\n    if not confirmed:\n        return []\n    start = confirmed[0][\'block_height\']\n    buckets = {}\n    for t in confirmed:\n        buckets.setdefault((t[\'block_height\'] - start) // window_blocks, []).append(t)\n    out = []\n    for k in sorted(buckets):\n        w_start = start + k * window_blocks\n        w_end = w_start + window_blocks - 1\n        window_txs = [t for t in confirmed if t[\'block_height\'] <= w_end] if cumulative else buckets[k]\n        out.append({\'block_start\': w_start, \'block_end\': w_end,\n                    \'n_txs\': len(buckets[k]), \'features\': compute_features(window_txs, address)})\n    return out\n', 'analyzer.py': '"""\nWallet risk analyzer: address -> sanctions check -> blockchain.com history ->\nElliptic++ features -> model score -> reasons -> plain-English summary.\n\nSettings come from environment variables (never hard-code keys):\n  MODEL_S3_URI            s3://bucket/elliptic/wallet_model/wallet_model.joblib\n  MODEL_PATH              local path (default: wallet_model.joblib)\n  BLOCKCHAIN_API_CODE     your blockchain.com key (optional)\n  BLOCKCHAIN_MIN_INTERVAL seconds between API calls (default 10)\n  MAX_TXS                 most recent transactions to fetch (default 200)\n  CACHE_DIR               local cache folder (default: cache)\n  CACHE_S3_URI            optional s3://bucket/prefix for a shared cache\n  BEDROCK_MODEL_ID        e.g. us.anthropic.claude-haiku-4-5-20251001-v1:0 (blank = rule-based only)\n  AWS_REGION              e.g. us-east-1\n"""\nimport os, re, json, time, math\nimport numpy as np\nimport pandas as pd\nimport requests\nimport joblib\nimport boto3\n\nfrom features import windowed_features, label\n\nCFG = {\n    \'model_s3_uri\': os.getenv(\'MODEL_S3_URI\', \'\'),\n    \'model_path\':   os.getenv(\'MODEL_PATH\', \'wallet_model.joblib\'),\n    \'api_code\':     os.getenv(\'BLOCKCHAIN_API_CODE\', \'\'),\n    \'min_interval\': float(os.getenv(\'BLOCKCHAIN_MIN_INTERVAL\', \'10\')),\n    \'max_txs\':      int(os.getenv(\'MAX_TXS\', \'200\')),\n    \'cache_dir\':    os.getenv(\'CACHE_DIR\', \'cache\'),\n    \'cache_s3_uri\': os.getenv(\'CACHE_S3_URI\', \'\'),\n    \'cache_ttl_h\':  float(os.getenv(\'CACHE_TTL_HOURS\', \'24\')),\n    \'bedrock_model\': os.getenv(\'BEDROCK_MODEL_ID\', \'\'),\n    \'region\':       os.getenv(\'AWS_REGION\', \'us-east-1\'),\n    \'ofac_url\':     os.getenv(\'OFAC_URL\', \'https://raw.githubusercontent.com/0xB10C/\'\n                          \'ofac-sanctioned-digital-currency-addresses/lists/sanctioned_addresses_XBT.txt\'),\n}\nAPI = \'https://blockchain.info/rawaddr/{address}\'\nPAGE = 50\n\n\n# ---------------------------------------------------------------- helpers\ndef _split_s3(uri):\n    b, _, k = uri[5:].partition(\'/\')\n    return b, k\n\n\ndef is_valid_address(address):\n    """Basic format check for legacy (1...), P2SH (3...) and bech32 (bc1...) addresses."""\n    a = address.strip()\n    return bool(re.fullmatch(r\'[13][a-km-zA-HJ-NP-Z1-9]{25,34}\', a) or\n                re.fullmatch(r\'bc1[02-9ac-hj-np-z]{11,87}\', a.lower()))\n\n\ndef normalize(address):\n    a = address.strip()\n    return a.lower() if a.lower().startswith(\'bc1\') else a\n\n\n# ---------------------------------------------------------------- model\n_bundle = None\ndef load_bundle():\n    global _bundle\n    if _bundle is None:\n        path = CFG[\'model_path\']\n        if not os.path.exists(path) and CFG[\'model_s3_uri\']:\n            b, k = _split_s3(CFG[\'model_s3_uri\'])\n            boto3.client(\'s3\', region_name=CFG[\'region\']).download_file(b, k, path)\n        _bundle = joblib.load(path)\n    return _bundle\n\n\n# ---------------------------------------------------------------- sanctions list\n_ofac = None\ndef ofac_list():\n    """OFAC-sanctioned Bitcoin addresses (None if the list couldn\'t be downloaded)."""\n    global _ofac\n    if _ofac is None:\n        try:\n            r = requests.get(CFG[\'ofac_url\'], timeout=15); r.raise_for_status()\n            _ofac = {line.strip() for line in r.text.splitlines() if line.strip()}\n        except Exception:\n            return None\n    return _ofac\n\n\n# ---------------------------------------------------------------- blockchain.com client\nclass BlockchainClient:\n    """Rate-limited, cached client for blockchain.com\'s free address endpoint."""\n    def __init__(self):\n        self._last = 0.0\n        os.makedirs(CFG[\'cache_dir\'], exist_ok=True)\n        self.s3 = boto3.client(\'s3\', region_name=CFG[\'region\']) if CFG[\'cache_s3_uri\'] else None\n\n    def _wait(self):\n        gap = CFG[\'min_interval\'] - (time.time() - self._last)\n        if gap > 0:\n            time.sleep(gap)\n        self._last = time.time()\n\n    def _get(self, address, offset):\n        params = {\'limit\': PAGE, \'offset\': offset}\n        if CFG[\'api_code\']:\n            params[\'api_code\'] = CFG[\'api_code\']\n        for attempt in range(4):\n            self._wait()\n            r = requests.get(API.format(address=address), params=params, timeout=30)\n            if r.status_code == 200:\n                return r.json()\n            if r.status_code in (429, 500, 502, 503, 504):\n                time.sleep(15 * (attempt + 1))      # back off when rate limited\n                continue\n            r.raise_for_status()\n        raise RuntimeError(\'blockchain.com kept rate-limiting; try again in a minute\')\n\n    # cache: local file first, then optional S3\n    def _cache_key(self, address):\n        return f"{address}_{CFG[\'max_txs\']}.json"\n\n    def _cache_read(self, address):\n        path = os.path.join(CFG[\'cache_dir\'], self._cache_key(address))\n        if not os.path.exists(path) and self.s3:\n            b, p = _split_s3(CFG[\'cache_s3_uri\'])\n            try:\n                self.s3.download_file(b, f"{p.rstrip(\'/\')}/{self._cache_key(address)}", path)\n            except Exception:\n                return None\n        if os.path.exists(path):\n            with open(path) as f:\n                data = json.load(f)\n            if time.time() - data.get(\'_fetched_at\', 0) < CFG[\'cache_ttl_h\'] * 3600:\n                return data\n        return None\n\n    def _cache_write(self, address, data):\n        path = os.path.join(CFG[\'cache_dir\'], self._cache_key(address))\n        with open(path, \'w\') as f:\n            json.dump(data, f)\n        if self.s3:\n            b, p = _split_s3(CFG[\'cache_s3_uri\'])\n            try:\n                self.s3.upload_file(path, b, f"{p.rstrip(\'/\')}/{self._cache_key(address)}")\n            except Exception:\n                pass\n\n    def fetch(self, address, progress=None):\n        cached = self._cache_read(address)\n        if cached:\n            if progress: progress(1.0, \'Loaded from cache\')\n            cached[\'_from_cache\'] = True\n            return cached\n        first = self._get(address, 0)\n        n_tx = first.get(\'n_tx\', 0)\n        target = min(n_tx, CFG[\'max_txs\'])\n        txs = list(first.get(\'txs\', []))\n        pages = max(1, math.ceil(target / PAGE))\n        if progress: progress(1 / pages, f\'Fetched page 1 of {pages}\')\n        while len(txs) < target:\n            page = self._get(address, len(txs)).get(\'txs\', [])\n            if not page:\n                break\n            txs.extend(page)\n            if progress: progress(min(1.0, len(txs) / target), f\'Fetched {len(txs)} of {target} transactions\')\n        data = {k: first.get(k) for k in (\'address\', \'n_tx\', \'total_received\', \'total_sent\', \'final_balance\')}\n        data[\'txs\'] = txs[:target]\n        data[\'_fetched_at\'] = time.time()\n        self._cache_write(address, data)\n        data[\'_from_cache\'] = False\n        return data\n\n\n# ---------------------------------------------------------------- scoring + reasons\ndef tier(p, thr):\n    return \'high\' if p >= thr else \'medium\' if p >= thr / 2 else \'low\'\n\n\ndef reasons(bundle, feats, n_risk=3, n_ok=2):\n    """Compare the wallet to typical illicit vs licit wallets on the model\'s most important features."""\n    ref = bundle[\'ref_stats\']\n    risk, ok = [], []\n    for f in bundle[\'importance_order\'][:20]:\n        r = ref[f]\n        v = math.log1p(max(feats[f], 0))\n        li, il = math.log1p(max(r[\'licit_median\'], 0)), math.log1p(max(r[\'illicit_median\'], 0))\n        if abs(li - il) < 0.05:           # feature doesn\'t separate the classes\n            continue\n        item = {\'feature\': f, \'label\': label(f), \'value\': feats[f],\n                \'illicit_median\': r[\'illicit_median\'], \'licit_median\': r[\'licit_median\']}\n        (risk if abs(v - il) < abs(v - li) else ok).append(item)\n    return risk[:n_risk], ok[:n_ok]\n\n\ndef out_of_range_share(bundle, feats):\n    ref = bundle[\'ref_stats\']\n    outside = [f for f in bundle[\'features\'] if not (ref[f][\'p01\'] <= feats[f] <= ref[f][\'p99\'])]\n    return len(outside) / len(bundle[\'features\'])\n\n\ndef _fmt(x):\n    return f\'{x:,.0f}\' if abs(x) >= 100 else f\'{x:,.4g}\'\n\n\ndef rule_summary(rep):\n    if rep[\'sanctioned\']:\n        return (\'This address appears on the U.S. Treasury OFAC sanctions list. Transacting with it may be \'\n                \'prohibited regardless of the model score.\')\n    if rep[\'status\'] != \'scored\':\n        return rep[\'message\']\n    lines = [f"Risk score {rep[\'risk_score\']:.0f}/100 ({rep[\'risk_tier\']} risk), based on "\n             f"{rep[\'n_txs_analyzed\']} recent transactions across {len(rep[\'windows\'])} two-week windows."]\n    if rep[\'risk_reasons\']:\n        lines.append(\'Patterns that resemble illicit wallets: \' + \'; \'.join(\n            f"{r[\'label\']} = {_fmt(r[\'value\'])} (typical illicit {_fmt(r[\'illicit_median\'])}, "\n            f"typical licit {_fmt(r[\'licit_median\'])})" for r in rep[\'risk_reasons\']) + \'.\')\n    if rep[\'ok_reasons\']:\n        lines.append(\'Patterns that resemble licit wallets: \' + \'; \'.join(r[\'label\'] for r in rep[\'ok_reasons\']) + \'.\')\n    if rep[\'out_of_range\'] > 0.3:\n        lines.append(\'Caution: this wallet behaves unlike most training wallets, so the score is less reliable.\')\n    lines.append(\'This is a risk indicator, not proof of wrongdoing.\')\n    return \' \'.join(lines)\n\n\nSYSTEM_PROMPT = (\n    \'You are an anti-money-laundering analyst assistant. Write a short, plain-English summary (max 120 words) \'\n    \'of a Bitcoin wallet risk report for a non-expert. Use only the facts given. Never state or imply the owner \'\n    \'is a criminal; describe risk indicators and patterns. Mention the main reasons, any sanctions match, and any \'\n    \'reliability caveats. End with one suggested next step for the user.\')\n\n\ndef bedrock_summary(rep):\n    facts = {k: rep[k] for k in (\'address\', \'status\', \'sanctioned\', \'risk_score\', \'risk_tier\', \'n_txs_analyzed\',\n                                 \'risk_reasons\', \'ok_reasons\', \'out_of_range\', \'message\') if k in rep}\n    client = boto3.client(\'bedrock-runtime\', region_name=CFG[\'region\'])\n    resp = client.converse(\n        modelId=CFG[\'bedrock_model\'],\n        system=[{\'text\': SYSTEM_PROMPT}],\n        messages=[{\'role\': \'user\', \'content\': [{\'text\': json.dumps(facts, default=float)}]}],\n        inferenceConfig={\'maxTokens\': 400, \'temperature\': 0.2})\n    return resp[\'output\'][\'message\'][\'content\'][0][\'text\'].strip()\n\n\n# ---------------------------------------------------------------- main entry point\ndef analyze(address, client=None, progress=None):\n    address = normalize(address)\n    rep = {\'address\': address, \'status\': \'scored\', \'message\': \'\', \'sanctioned\': False,\n           \'sanctions_list_loaded\': True, \'windows\': [], \'risk_reasons\': [], \'ok_reasons\': [],\n           \'out_of_range\': 0.0, \'summary_source\': \'rules\'}\n\n    if not is_valid_address(address):\n        rep.update(status=\'invalid\', message=\'That does not look like a valid Bitcoin address.\')\n        rep[\'summary\'] = rule_summary(rep); return rep\n\n    ofac = ofac_list()\n    rep[\'sanctions_list_loaded\'] = ofac is not None\n    rep[\'sanctioned\'] = bool(ofac and address in ofac)\n\n    bundle = load_bundle()\n    data = (client or BlockchainClient()).fetch(address, progress)\n    txs = [t for t in data[\'txs\'] if t.get(\'block_height\') is not None]\n    rep.update(n_tx_total=data.get(\'n_tx\', 0), n_txs_analyzed=len(txs), from_cache=data.get(\'_from_cache\', False),\n               final_balance_btc=(data.get(\'final_balance\') or 0) / 1e8)\n\n    if len(txs) < 2:\n        rep.update(status=\'insufficient\', message=f\'Only {len(txs)} confirmed transaction(s) found, which is not \'\n                   \'enough history to score this wallet reliably.\')\n    else:\n        wins = windowed_features(txs, address, bundle[\'window_blocks\'], bundle[\'cumulative\'])\n        X = pd.DataFrame([w[\'features\'] for w in wins])[bundle[\'features\']]\n        probs = bundle[\'model\'].predict_proba(X)[:, 1]\n        worst = int(np.argmax(probs))\n        thr = bundle[\'threshold\']\n        for w, p in zip(wins, probs):\n            w[\'risk\'] = float(p)\n        rep[\'windows\'] = [{k: w[k] for k in (\'block_start\', \'block_end\', \'n_txs\', \'risk\')} for w in wins]\n        rep[\'risk_prob\'] = float(probs[worst])\n        rep[\'risk_score\'] = round(float(probs[worst]) * 100, 1)\n        rep[\'risk_tier\'] = tier(float(probs[worst]), thr)\n        rep[\'worst_window\'] = rep[\'windows\'][worst]\n        rep[\'worst_features\'] = wins[worst][\'features\']\n        rep[\'risk_reasons\'], rep[\'ok_reasons\'] = reasons(bundle, wins[worst][\'features\'])\n        rep[\'out_of_range\'] = out_of_range_share(bundle, wins[worst][\'features\'])\n        if data.get(\'n_tx\', 0) > len(data[\'txs\']):\n            rep[\'message\'] = (f"Analyzed the most recent {len(data[\'txs\'])} of {data[\'n_tx\']:,} transactions.")\n\n    if rep[\'sanctioned\']:\n        rep[\'risk_tier\'] = \'sanctioned\'\n\n    rep[\'summary\'] = rule_summary(rep)\n    if CFG[\'bedrock_model\']:\n        try:\n            rep[\'summary\'] = bedrock_summary(rep)\n            rep[\'summary_source\'] = \'bedrock\'\n        except Exception as e:                  # no access / wrong model id -> keep rule-based text\n            rep[\'bedrock_error\'] = str(e)[:200]\n    return rep\n', 'app.py': '"""Streamlit web page: paste a Bitcoin address, get a risk report.  Run: streamlit run app.py"""\nimport os\nimport pandas as pd\nimport streamlit as st\nimport analyzer\nfrom features import label\n\nst.set_page_config(page_title=\'Wallet Risk Check\', page_icon=\'🔎\', layout=\'centered\')\n\nTIER_STYLE = {\'high\': (\'#b42318\', \'HIGH RISK\'), \'medium\': (\'#b54708\', \'MEDIUM RISK\'),\n              \'low\': (\'#067647\', \'LOW RISK\'), \'sanctioned\': (\'#7a0000\', \'SANCTIONED\')}\n\n\n@st.cache_resource\ndef model():\n    return analyzer.load_bundle()\n\n\n@st.cache_resource\ndef client():\n    return analyzer.BlockchainClient()\n\n\n@st.cache_data\ndef demo_addresses():\n    path = \'demo_addresses.csv\'\n    return pd.read_csv(path) if os.path.exists(path) else None\n\n\nbundle = model()\nst.title(\'🔎 Bitcoin Wallet Risk Check\')\nst.caption(\'Enter a Bitcoin address you are unsure about. We analyze its public transaction history \'\n           \'with a model trained on labeled wallets from the Elliptic++ dataset.\')\n\n# sidebar: model facts + demo addresses\nwith st.sidebar:\n    st.subheader(\'Model\')\n    m = bundle[\'metrics\'][bundle[\'model_name\']]\n    st.write(f"**{bundle[\'model_name\']}** trained on Elliptic++ wallets")\n    st.write(f"Test F1 {m[\'Illicit F1\']:.2f} · Precision {m[\'Illicit Precision\']:.2f} · Recall {m[\'Illicit Recall\']:.2f}")\n    demos = demo_addresses()\n    if demos is not None:\n        st.subheader(\'Try an example\')\n        for _, r in demos.groupby(\'kind\').head(3).iterrows():\n            if st.button(f"{r[\'kind\']}: {r[\'address\'][:14]}…", key=r[\'address\']):\n                st.session_state[\'addr\'] = r[\'address\']\n\naddress = st.text_input(\'Bitcoin address\', key=\'addr\', placeholder=\'1..., 3... or bc1...\')\ngo = st.button(\'Analyze\', type=\'primary\', disabled=not address)\n\nif go and address:\n    if not analyzer.is_valid_address(address):\n        st.error(\'That does not look like a valid Bitcoin address.\')\n        st.stop()\n    bar = st.progress(0.0, text=\'Fetching transaction history from blockchain.com (free tier ≈ 10 s per page)…\')\n    try:\n        rep = analyzer.analyze(address, client=client(),\n                               progress=lambda frac, msg: bar.progress(min(frac, 1.0), text=msg))\n    except Exception as e:\n        bar.empty()\n        st.error(f\'Could not fetch data for this address: {e}\')\n        st.stop()\n    bar.empty()\n\n    # headline card\n    tier = rep.get(\'risk_tier\', \'low\') if rep[\'status\'] == \'scored\' or rep[\'sanctioned\'] else None\n    if tier:\n        color, text = TIER_STYLE[tier]\n        score = f"{rep[\'risk_score\']:.0f}/100" if \'risk_score\' in rep else \'\'\n        st.markdown(f"<div style=\'padding:16px;border-radius:10px;background:{color};color:white\'>"\n                    f"<div style=\'font-size:14px;opacity:.85\'>Risk assessment</div>"\n                    f"<div style=\'font-size:30px;font-weight:700\'>{text} {score}</div></div>",\n                    unsafe_allow_html=True)\n    else:\n        st.warning(rep[\'message\'])\n\n    if rep[\'sanctioned\']:\n        st.error(\'⚠️ This address is on the U.S. Treasury OFAC sanctions list.\')\n    elif not rep[\'sanctions_list_loaded\']:\n        st.info(\'Sanctions list could not be loaded right now, so no sanctions check was done.\')\n\n    st.subheader(\'Summary\')\n    st.write(rep[\'summary\'])\n    st.caption(\'Written by Amazon Bedrock\' if rep[\'summary_source\'] == \'bedrock\' else \'Rule-based summary\')\n\n    if rep[\'status\'] == \'scored\':\n        c1, c2, c3 = st.columns(3)\n        c1.metric(\'Transactions analyzed\', rep[\'n_txs_analyzed\'])\n        c2.metric(\'Total on-chain transactions\', f"{rep[\'n_tx_total\']:,}")\n        c3.metric(\'Current balance (BTC)\', f"{rep[\'final_balance_btc\']:.4f}")\n        if rep[\'message\']:\n            st.caption(rep[\'message\'])\n\n        if rep[\'risk_reasons\'] or rep[\'ok_reasons\']:\n            st.subheader(\'Why\')\n            for r in rep[\'risk_reasons\']:\n                st.markdown(f"🔴 **{r[\'label\']}**: {r[\'value\']:,.4g} "\n                            f"(typical illicit {r[\'illicit_median\']:,.4g} · typical licit {r[\'licit_median\']:,.4g})")\n            for r in rep[\'ok_reasons\']:\n                st.markdown(f"🟢 **{r[\'label\']}**: {r[\'value\']:,.4g} "\n                            f"(typical illicit {r[\'illicit_median\']:,.4g} · typical licit {r[\'licit_median\']:,.4g})")\n\n        if len(rep[\'windows\']) > 1:\n            st.subheader(\'Risk over time (two-week windows)\')\n            w = pd.DataFrame(rep[\'windows\'])\n            w[\'risk score\'] = w[\'risk\'] * 100\n            st.line_chart(w.set_index(\'block_start\')[\'risk score\'])\n            st.caption(\'Each point is ~2016 blocks (about two weeks). The headline score is the riskiest window.\')\n\n        if rep[\'out_of_range\'] > 0.3:\n            st.warning(\'This wallet behaves unlike most wallets the model was trained on, so treat the score with caution.\')\n\n        with st.expander(\'All features for the riskiest window\'):\n            f = pd.Series(rep[\'worst_features\'])[bundle[\'features\']]\n            st.dataframe(pd.DataFrame({\'feature\': [label(k) for k in f.index], \'value\': f.values}),\n                         hide_index=True, width=\'stretch\')\n\n    with st.expander(\'About this tool and its limits\'):\n        st.markdown(\n            \'- Trained on Elliptic++ labeled wallets from 2016–2017. Criminal behavior changes over time, \'\n            \'so accuracy on today’s wallets will be lower than the test score.\\n\'\n            \'- Only the most recent transactions are analyzed (free API rate limit).\\n\'\n            \'- A risk score is an indicator for further review, **not proof of wrongdoing**.\\n\'\n            \'- Sanctions data: OFAC SDN list of Bitcoin addresses.\')\n        if rep.get(\'from_cache\'):\n            st.caption(\'Result used cached blockchain data (refreshed every 24 h).\')\n', 'setup_ec2.sh': '#!/bin/bash\n# One-time setup on a fresh Amazon Linux 2023 EC2 instance.\n# Usage: bash setup_ec2.sh your-bucket-name\nset -e\nBUCKET="$1"\nif [ -z "$BUCKET" ]; then echo "Usage: bash setup_ec2.sh your-bucket-name"; exit 1; fi\n\necho "== Installing Python 3.11"\nsudo dnf install -y python3.11 python3.11-pip\n\necho "== Downloading app code and model from S3"\nmkdir -p ~/app && cd ~/app\naws s3 cp "s3://$BUCKET/elliptic/app/" . --recursive\naws s3 cp "s3://$BUCKET/elliptic/wallet_model/" . --recursive\n\necho "== Installing Python packages (pinned to the versions used for training)"\npython3.11 -m venv venv\n./venv/bin/pip install --upgrade pip -q\n./venv/bin/pip install -r requirements_app.txt -q\n\nsed -i "s/^BUCKET=.*/BUCKET=$BUCKET/" config.env\necho "== Done. Check config.env (nano config.env), then start the app with: bash run_app.sh"\n', 'run_app.sh': '#!/bin/bash\n# Start (or restart) the web app in the background. Usage: bash run_app.sh\ncd ~/app\nset -a; source config.env; set +a\nexport MODEL_PATH=wallet_model.joblib\nexport CACHE_S3_URI="s3://$BUCKET/elliptic/cache/"\n\nif [ -f app.pid ]; then kill "$(cat app.pid)" 2>/dev/null || true; sleep 1; fi   # stop old copy\nnohup ./venv/bin/streamlit run app.py --server.port 8501 --server.address 0.0.0.0 \\\n      --server.headless true > app.log 2>&1 &\necho $! > app.pid\nsleep 5\nTOKEN=$(curl -s -X PUT "http://169.254.169.254/latest/api/token" -H "X-aws-ec2-metadata-token-ttl-seconds: 60")\nIP=$(curl -s -H "X-aws-ec2-metadata-token: $TOKEN" http://169.254.169.254/latest/meta-data/public-ipv4)\necho "App running. Open: http://$IP:8501"\necho "Logs: tail -f ~/app/app.log"\n', 'config.env': "# Settings for the wallet risk web app. Edit the values, then run: bash run_app.sh\n# (This file stays on the server. Never commit it to GitHub.)\n\n# your bucket name\nBUCKET=sagemaker-notebook-469627110704\n\n# region your EC2 instance and Bedrock are in\nAWS_REGION=us-east-1\n\n# blockchain.com free API key (leave blank if you don't have one)\nBLOCKCHAIN_API_CODE=\n\n# seconds between blockchain.com calls (free tier: keep at 10)\nBLOCKCHAIN_MIN_INTERVAL=10\n\n# most recent transactions to analyze per wallet\nMAX_TXS=200\n\n# Bedrock model ID for the AI summary (copy from Bedrock console). Blank = rule-based summary only.\nBEDROCK_MODEL_ID=us.anthropic.claude-haiku-4-5-20251001-v1:0\n"}
os.makedirs('app_files', exist_ok=True)
for name, text in APP_FILES.items():
    with open(f'app_files/{name}', 'w') as f: f.write(text)
    s3.upload_file(f'app_files/{name}', BUCKET, f'elliptic/app/{name}')
    print(f'uploaded s3://{BUCKET}/elliptic/app/{name}')

In [ ]:
# 11. PARITY CHECK: rebuild features from live blockchain.com data for a few dataset addresses
#     and compare with the dataset's values. If these match, live scores are trustworthy.
import importlib, features; importlib.reload(features)          # picks up an updated features.py
compute_features = features.compute_features
def parity(address, row, max_txs=50):
    r = requests.get(f'https://blockchain.info/rawaddr/{address}', params={'limit': max_txs}, timeout=30)
    r.raise_for_status(); d = r.json()
    if d['n_tx'] > max_txs:
        return None                                   # history too long to check with one free call
    lo, hi = int(row['first_block_appeared_in']), int(row['last_block_appeared_in'])
    txs = [t for t in d['txs'] if t.get('block_height') and (t['block_height'] <= hi) and
           (CUMULATIVE or t['block_height'] >= lo)]
    live = compute_features(txs, address)
    return pd.DataFrame({'dataset': row[FEATURES].astype(float), 'live': pd.Series(live)[FEATURES]})

checked, results_parity = 0, []
for _, row in df[df.y == 1].sample(frac=1, random_state=SEED).iterrows():
    if checked >= 3: break
    try:
        comp = parity(row['address'], row)
    except Exception as e:
        print('skip', row['address'], e); time.sleep(10); continue
    time.sleep(10)                                     # free tier: ~1 call / 10 s
    if comp is None: continue
    checked += 1
    close = np.isclose(comp['dataset'], comp['live'], rtol=0.02, atol=1e-8)
    results_parity.append(close.mean())
    print(f"{row['address']} (time step {row['time_step']}): {close.mean():.0%} of features match")
    print(comp[~close].to_string() if (~close).any() else '(all match)', '\n')
print('average match rate:', np.mean(results_parity) if results_parity else 'n/a')